# 04. DEG可視化

DEG結果をインタラクティブに可視化します。

**カーネル**: RNA-seq (Python) / rnaseq_env  
**入力**: `results/deg_*.csv`, `results/count_matrix.csv`, `sample_metadata.csv`  
**出力**: Volcano Plot, サンプルレベル発現ヒートマップ, DEGステータスヒートマップ (HTML)

### ヒートマップの種類
1. **サンプルレベル発現ヒートマップ** — 全サンプルの正規化発現量（Z-score）を表示。遺伝子・サンプル両軸でクラスタリングし、特定のサンプル群で共通に変動する遺伝子パターンを発見しやすい設計。
2. **ペアワイズ比較DEGステータスヒートマップ** — 各比較条件でのUp/Down/NS状態を一覧表示。

### 色の定義
- **赤色**: 有意に増加 (log2FC > 1.0 & padj < 0.05)
- **グレー**: 変化なし (padj >= 0.05)
- **青色**: 有意に減少 (log2FC < -1.0 & padj < 0.05)

## 設定

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
import glob, os

# === 設定 ===
LFC_THRESHOLD = 1.0
PADJ_THRESHOLD = 0.05

# 色の定義
COLOR_UP = "#E74C3C"     # 赤: 増加
COLOR_DOWN = "#3498DB"   # 青: 減少
COLOR_NS = "#BDC3C7"     # グレー: 変化なし

print("設定完了")

## データ読み込みとDEG分類

In [ ]:
# 全比較結果を読み込み
result_files = sorted(glob.glob("results/deg_*_vs_*.csv"))

if not result_files:
    raise FileNotFoundError(
        "DEG結果ファイルが見つかりません: results/deg_*_vs_*.csv\n"
        "先に 03_DEG_Analysis.ipynb を実行してください。"
    )

all_results = {}

for f in result_files:
    df = pd.read_csv(f)
    if "comparison" not in df.columns:
        raise ValueError(f"'comparison' カラムが見つかりません: {f}")
    comp_name = df["comparison"].iloc[0]
    all_results[comp_name] = df
    
print(f"比較条件数: {len(all_results)}")
for name, df in all_results.items():
    print(f"  {name}: {len(df)} genes")

In [ ]:
def classify_deg(row):
    if pd.isna(row["padj"]) or pd.isna(row["log2FoldChange"]):
        return "NS"
    if row["padj"] < PADJ_THRESHOLD and row["log2FoldChange"] > LFC_THRESHOLD:
        return "Up"
    elif row["padj"] < PADJ_THRESHOLD and row["log2FoldChange"] < -LFC_THRESHOLD:
        return "Down"
    return "NS"

for comp_name, df in all_results.items():
    df["deg_class"] = df.apply(classify_deg, axis=1)
    n_up = (df["deg_class"] == "Up").sum()
    n_down = (df["deg_class"] == "Down").sum()
    n_ns = (df["deg_class"] == "NS").sum()
    print(f"{comp_name}: Up={n_up}, Down={n_down}, NS={n_ns}")

## Volcano Plot（インタラクティブ・ドロップダウン切替）

In [ ]:
fig = go.Figure()
comp_names = list(all_results.keys())

for i, (comp_name, df) in enumerate(all_results.items()):
    df = df.dropna(subset=["log2FoldChange", "padj"]).copy()
    df["-log10padj"] = -np.log10(df["padj"].clip(lower=1e-300))
    
    for deg_class, color in [("NS", COLOR_NS), ("Down", COLOR_DOWN), ("Up", COLOR_UP)]:
        subset = df[df["deg_class"] == deg_class]
        fig.add_trace(go.Scattergl(
            x=subset["log2FoldChange"],
            y=subset["-log10padj"],
            mode="markers",
            marker=dict(color=color, size=4, opacity=0.6),
            name=deg_class,
            text=subset["gene_id"],
            hovertemplate="<b>%{text}</b><br>log2FC: %{x:.2f}<br>-log10(padj): %{y:.2f}<extra></extra>",
            visible=(i == 0),
            legendgroup=deg_class,
            showlegend=(i == 0)
        ))

# ドロップダウン
buttons = []
for i, name in enumerate(comp_names):
    vis = [False] * (len(comp_names) * 3)
    for j in range(3):
        vis[i * 3 + j] = True
    buttons.append(dict(label=name, method="update",
                        args=[{"visible": vis}, {"title": f"Volcano Plot: {name}"}]))

# 閾値線
fig.add_hline(y=-np.log10(PADJ_THRESHOLD), line_dash="dash", line_color="grey", opacity=0.5)
fig.add_vline(x=LFC_THRESHOLD, line_dash="dash", line_color="grey", opacity=0.5)
fig.add_vline(x=-LFC_THRESHOLD, line_dash="dash", line_color="grey", opacity=0.5)

fig.update_layout(
    updatemenus=[dict(type="dropdown", x=0.0, y=1.15, showactive=True, buttons=buttons)],
    title=f"Volcano Plot: {comp_names[0]}",
    xaxis_title="log2 Fold Change",
    yaxis_title="-log10(adjusted p-value)",
    template="plotly_white", width=900, height=700,
    legend=dict(title="DEG Classification")
)

fig.show()

In [ ]:
# HTML出力
fig.write_html("results/volcano_plot_interactive.html", include_plotlyjs=True)
print("出力: results/volcano_plot_interactive.html")

## サンプルレベル発現ヒートマップ

全サンプルの正規化発現量（Z-score）をヒートマップで表示します。
DEGとして検出された遺伝子を対象に、**遺伝子軸・サンプル軸の両方で階層的クラスタリング**を行い、
特定のサンプル群で共通に変動する遺伝子パターンを視覚的に把握できます。

### 正規化手順
1. **log2(CPM + 1)** — ライブラリサイズ補正 + 対数変換
2. **Z-score（行方向）** — 遺伝子ごとに平均0・標準偏差1に標準化

### クラスタリング
- **遺伝子（行）**: Ward法 — 類似した発現パターンの遺伝子がまとまる
- **サンプル（列）**: Ward法 — 類似した発現プロファイルのサンプルがまとまる
- 上部に**条件カラーバー**を表示し、サンプルがどの実験条件に属するか一目で判別可能

In [ ]:
# === カウント行列・メタデータの読み込み ===
if not os.path.exists("results/count_matrix.csv"):
    raise FileNotFoundError(
        "results/count_matrix.csv が見つかりません。\n"
        "先に 02_Mapping_and_Counting.ipynb を実行してください。"
    )
if not os.path.exists("sample_metadata.csv"):
    raise FileNotFoundError(
        "sample_metadata.csv が見つかりません。\n"
        "README.md のセットアップ手順を確認してください。"
    )

count_matrix = pd.read_csv("results/count_matrix.csv", index_col=0)
sample_meta = pd.read_csv("sample_metadata.csv")

# サンプル順を合わせる
count_matrix = count_matrix[sample_meta["sample_id"].tolist()]

# === DEG遺伝子の抽出（全比較のunion） ===
deg_genes = set()
for comp_name, df in all_results.items():
    degs = df[df["deg_class"].isin(["Up", "Down"])]["gene_id"].tolist()
    deg_genes.update(degs)

# カウント行列に存在するDEG遺伝子のみ
deg_genes = sorted(deg_genes & set(count_matrix.index))

if len(deg_genes) == 0:
    print("警告: DEG遺伝子が0件のため、サンプルレベルヒートマップはスキップされます。")
else:
    print(f"DEG遺伝子数（全比較のunion）: {len(deg_genes)}")
    print(f"サンプル数: {len(count_matrix.columns)}")
    print(f"条件: {', '.join(sample_meta['condition'].unique())}")

In [ ]:
if len(deg_genes) == 0:
    print("DEGが0件のため、サンプルレベルヒートマップの生成をスキップします。")
else:
    from plotly.subplots import make_subplots

    # === 正規化: log2(CPM+1) → Z-score ===
    deg_counts = count_matrix.loc[deg_genes]

    # CPM (Counts Per Million)
    lib_sizes = deg_counts.sum(axis=0)
    cpm = deg_counts.div(lib_sizes, axis=1) * 1e6
    log2cpm = np.log2(cpm + 1)

    # Z-score（行方向: 遺伝子ごと）
    row_mean = log2cpm.mean(axis=1)
    row_std = log2cpm.std(axis=1)
    # 標準偏差0の遺伝子を除外
    valid = row_std > 0
    log2cpm = log2cpm[valid]
    zscore = log2cpm.sub(row_mean[valid], axis=0).div(row_std[valid], axis=0)

    print(f"Z-score正規化後の遺伝子数: {len(zscore)}")

    # === 両軸の階層的クラスタリング ===
    # 遺伝子（行）クラスタリング
    if len(zscore) > 1:
        gene_dist = pdist(zscore.values, metric="correlation")
        gene_link = linkage(gene_dist, method="ward")
        gene_order = leaves_list(gene_link)
    else:
        gene_order = [0]

    # サンプル（列）クラスタリング
    if zscore.shape[1] > 1:
        sample_dist = pdist(zscore.values.T, metric="correlation")
        sample_link = linkage(sample_dist, method="ward")
        sample_order = leaves_list(sample_link)
    else:
        sample_order = [0]

    # クラスタリング順に並び替え
    zscore_ordered = zscore.iloc[gene_order, sample_order]
    ordered_samples = zscore_ordered.columns.tolist()

    # === 条件カラーバーの構築 ===
    sample_to_condition = dict(zip(sample_meta["sample_id"], sample_meta["condition"]))
    conditions_unique = sample_meta["condition"].unique().tolist()

    # 条件ごとの色パレット（最大10条件まで対応）
    condition_palette = [
        "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
        "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4"
    ]
    condition_colors = {c: condition_palette[i % len(condition_palette)]
                        for i, c in enumerate(conditions_unique)}

    # 各サンプルの条件色
    color_bar_values = [condition_colors[sample_to_condition[s]] for s in ordered_samples]
    condition_labels = [sample_to_condition[s] for s in ordered_samples]

    # === ヒートマップ + カラーバー（subplot） ===
    fig_expr = make_subplots(
        rows=2, cols=1,
        row_heights=[0.03, 0.97],
        vertical_spacing=0.0,
        shared_xaxes=True
    )

    # 条件カラーバー（上段）
    # 数値にマッピングしてヒートマップとして描画
    cond_numeric = [conditions_unique.index(sample_to_condition[s]) for s in ordered_samples]
    cond_colorscale = []
    n_cond = len(conditions_unique)
    for i, c in enumerate(conditions_unique):
        lo = i / max(n_cond, 1)
        hi = (i + 1) / max(n_cond, 1)
        cond_colorscale.append([lo, condition_colors[c]])
        cond_colorscale.append([hi, condition_colors[c]])

    fig_expr.add_trace(go.Heatmap(
        z=[cond_numeric],
        x=ordered_samples,
        colorscale=cond_colorscale,
        zmin=0, zmax=max(n_cond - 1, 1),
        showscale=False,
        hovertext=[[f"Sample: {s}<br>Condition: {sample_to_condition[s]}" for s in ordered_samples]],
        hoverinfo="text"
    ), row=1, col=1)

    # ホバーテキスト
    hover_expr = []
    for gene in zscore_ordered.index:
        row_hover = []
        for sample in ordered_samples:
            row_hover.append(
                f"Gene: {gene}<br>"
                f"Sample: {sample}<br>"
                f"Condition: {sample_to_condition[sample]}<br>"
                f"Z-score: {zscore_ordered.loc[gene, sample]:.2f}"
            )
        hover_expr.append(row_hover)

    # Z-scoreの範囲をクリップ（外れ値対策）
    z_clipped = zscore_ordered.clip(-3, 3).values

    # メインヒートマップ（下段）— 青(-3) → 白(0) → 赤(+3)
    fig_expr.add_trace(go.Heatmap(
        z=z_clipped,
        x=ordered_samples,
        y=zscore_ordered.index.tolist(),
        colorscale=[
            [0.0, "#3498DB"], [0.5, "#FFFFFF"], [1.0, "#E74C3C"]
        ],
        zmin=-3, zmax=3,
        text=hover_expr, hoverinfo="text",
        colorbar=dict(
            title="Z-score",
            len=0.7, y=0.45,
            tickvals=[-3, -2, -1, 0, 1, 2, 3]
        )
    ), row=2, col=1)

    # レイアウト
    n_genes = len(zscore_ordered)
    fig_h = max(700, min(3000, n_genes * 10 + 300))

    fig_expr.update_layout(
        title=dict(
            text="サンプルレベル発現ヒートマップ（DEG遺伝子 / Z-score）",
            font=dict(size=16)
        ),
        template="plotly_white",
        width=max(700, len(ordered_samples) * 80 + 300),
        height=fig_h,
    )

    # 軸設定
    fig_expr.update_xaxes(showticklabels=False, row=1, col=1)
    fig_expr.update_yaxes(showticklabels=False, row=1, col=1)
    fig_expr.update_xaxes(
        title_text="サンプル", tickangle=45, row=2, col=1
    )
    fig_expr.update_yaxes(
        title_text="遺伝子",
        tickfont=dict(size=7 if n_genes > 80 else (8 if n_genes > 40 else 10)),
        row=2, col=1
    )

    # 条件カラー凡例をアノテーションで追加
    legend_text = "  ".join(
        [f'<span style="color:{condition_colors[c]};">&#9632;</span> {c}'
         for c in conditions_unique]
    )
    fig_expr.add_annotation(
        text=legend_text,
        xref="paper", yref="paper",
        x=0.5, y=1.06, showarrow=False,
        font=dict(size=12),
        align="center"
    )

    fig_expr.show()

In [ ]:
if len(deg_genes) > 0:
    fig_expr.write_html("results/expression_heatmap_interactive.html", include_plotlyjs=True)
    print("出力: results/expression_heatmap_interactive.html")
else:
    print("DEGが0件のため、サンプルレベルヒートマップHTML出力をスキップしました。")

## ペアワイズ比較 DEGステータスヒートマップ

全ての比較条件で、各遺伝子のDEG状態（Up / NS / Down）を一覧表示します。
サンプルレベルヒートマップと組み合わせて、どの比較で有意と判定されたかを確認できます。

- **赤色**: 増加 (Up-regulated)
- **グレー**: 変化なし (Not Significant)
- **青色**: 減少 (Down-regulated)

In [ ]:
# DEG分類行列の構築
deg_matrices = {}
lfc_matrices = {}

for comp_name, df in all_results.items():
    df_idx = df.set_index("gene_id")
    deg_matrices[comp_name] = df_idx["deg_class"].map({"Up": 1, "NS": 0, "Down": -1}).fillna(0)
    lfc_matrices[comp_name] = df_idx["log2FoldChange"].fillna(0)

deg_df = pd.DataFrame(deg_matrices)
lfc_df = pd.DataFrame(lfc_matrices)

# いずれかの比較でDEGだった遺伝子のみ
is_deg = (deg_df.abs() > 0).any(axis=1)
deg_df = deg_df[is_deg]
lfc_df = lfc_df.loc[deg_df.index]

if len(deg_df) == 0:
    print("警告: いずれの条件でもDEGが検出されませんでした。ステータスヒートマップはスキップされます。")
else:
    print(f"DEG遺伝子数 (いずれかの条件): {len(deg_df)}")

In [ ]:
if len(deg_df) == 0:
    print("DEGが0件のため、ステータスヒートマップの生成をスキップします。")
else:
    # クラスタリング（2遺伝子以上の場合のみ）
    if len(deg_df) > 1:
        dist_matrix = pdist(deg_df.values, metric="euclidean")
        link = linkage(dist_matrix, method="ward")
        order = leaves_list(link)
        deg_df = deg_df.iloc[order]
        lfc_df = lfc_df.iloc[order]

    # ホバーテキスト
    hover = []
    for gene in deg_df.index:
        row = []
        for comp in deg_df.columns:
            s = {1: "Up", 0: "NS", -1: "Down"}.get(deg_df.loc[gene, comp], "NS")
            row.append(f"Gene: {gene}<br>Comparison: {comp}<br>Status: {s}<br>log2FC: {lfc_df.loc[gene, comp]:.2f}")
        hover.append(row)

    # カラースケール: 青(-1) → グレー(0) → 赤(+1)
    colorscale = [
        [0.0, COLOR_DOWN], [0.45, COLOR_DOWN],
        [0.45, COLOR_NS], [0.55, COLOR_NS],
        [0.55, COLOR_UP], [1.0, COLOR_UP]
    ]

    fig_h = max(600, min(2000, len(deg_df) * 12 + 200))

    fig_heat = go.Figure(data=go.Heatmap(
        z=deg_df.values,
        x=deg_df.columns.tolist(),
        y=deg_df.index.tolist(),
        colorscale=colorscale,
        zmin=-1, zmax=1,
        text=hover, hoverinfo="text",
        colorbar=dict(title="DEG Status", tickvals=[-1, 0, 1], ticktext=["Down", "NS", "Up"], len=0.5)
    ))

    fig_heat.update_layout(
        title="ペアワイズ比較 DEGステータスヒートマップ",
        xaxis_title="比較条件",
        yaxis_title="遺伝子",
        template="plotly_white",
        width=max(600, len(deg_df.columns) * 150 + 300),
        height=fig_h,
        yaxis=dict(tickfont=dict(size=8 if len(deg_df) > 50 else 10))
    )

    fig_heat.show()

In [ ]:
if len(deg_df) > 0:
    fig_heat.write_html("results/deg_heatmap_interactive.html", include_plotlyjs=True)
    print("出力: results/deg_heatmap_interactive.html")
else:
    print("DEGが0件のため、ステータスヒートマップHTML出力をスキップしました。")

## DEG数サマリー (バープロット)

In [ ]:
counts_list = []
for comp_name, df in all_results.items():
    n_up = (df["deg_class"] == "Up").sum()
    n_down = (df["deg_class"] == "Down").sum()
    counts_list.append({"comparison": comp_name, "Up": n_up, "Down": -n_down})

count_df = pd.DataFrame(counts_list)

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(x=count_df["comparison"], y=count_df["Up"],
                         name="Up-regulated", marker_color=COLOR_UP))
fig_bar.add_trace(go.Bar(x=count_df["comparison"], y=count_df["Down"],
                         name="Down-regulated", marker_color=COLOR_DOWN))

fig_bar.update_layout(
    title="DEG数 (各比較条件)",
    xaxis_title="比較", yaxis_title="遺伝子数",
    barmode="relative", template="plotly_white"
)
fig_bar.show()

## 完了

### 生成されたファイル
- `results/volcano_plot_interactive.html` — インタラクティブVolcano Plot
- `results/expression_heatmap_interactive.html` — サンプルレベル発現ヒートマップ（Z-score、両軸クラスタリング）
- `results/deg_heatmap_interactive.html` — ペアワイズ比較DEGステータスヒートマップ

HTMLファイルをブラウザで開くと、インタラクティブに操作できます。

### 操作方法
- **Volcano Plot**: ドロップダウンで比較条件を切替。マウスホバーで遺伝子名表示。
- **サンプルレベルヒートマップ**: 遺伝子×サンプルのZ-scoreを表示。両軸Ward法クラスタリング済み。上部カラーバーで条件を色分け。マウスホバーで遺伝子名・サンプル名・条件・Z-scoreを表示。
- **DEGステータスヒートマップ**: マウスホバーで遺伝子名・条件・log2FC・DEGステータスを表示。ズーム・パン可能。